<a href="https://colab.research.google.com/github/Moquiuti/LangChainePython/blob/main/RAG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install -q langchain langchain-community langchain-google-genai pypdf faiss-cpu python-dotenv

In [3]:
from google.colab import userdata
import os

api_key = userdata.get("python-ai-integrada")
os.environ["GOOGLE_API_KEY"] = api_key

print("Chave carregada com sucesso!" if api_key else "Chave não encontrada.")

Chave carregada com sucesso!


In [4]:
from dotenv import load_dotenv
import os

from langchain_community.document_loaders import PyPDFLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

In [6]:
arquivos_pdf = [
    "/content/documentos/GTB_standard_Nov23.pdf",
    "/content/documentos/GTB_gold_Nov23.pdf",
    "/content/documentos/GTB_platinum_Nov23.pdf",
]

documentos = []
for caminho in arquivos_pdf:
    loader = PyPDFLoader(caminho)
    documentos.extend(loader.load())

print(f"Total de documentos carregados: {len(documentos)}")

Total de documentos carregados: 59


In [7]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

chunks = splitter.split_documents(documentos)

print(f"Total de chunks gerados: {len(chunks)}")
print(chunks[0].page_content[:500])

Total de chunks gerados: 217
1 
 
 
Programa de Cartão da Edição MasterCard Standard 
 
Informações importantes. Leia e guarde as informações. 
 
Este Guia de Benefícios contém informações detalhadas sobre serviços aos quais você terá acesso como 
portador de cartão preferencial. Esses benefícios e serviços estão em vigor para portadores do cartão 
MasterCard Standard qualificados à partir de 1 de novembro de 2023. Este Guia substitui qualquer guia 
ou comunicação de programa que você recebeu anteriormente. 
 
As informaçõe


In [8]:
embeddings = GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-001"
)

In [22]:
vectorstore = FAISS.from_documents(
    documents=chunks,
    embedding=embeddings
)

print("Vector store criado com sucesso.")

GoogleGenerativeAIError: Error embedding content (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/embed_content_free_tier_requests, limit: 100, model: gemini-embedding-1.0\nPlease retry in 56.484686325s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/embed_content_free_tier_requests', 'quotaId': 'EmbedContentRequestsPerMinutePerUserPerProjectPerModel-FreeTier', 

In [12]:
retriever = vectorstore.as_retriever(
    search_kwargs={"k": 4}
)

NameError: name 'vectorstore' is not defined

In [13]:
consulta = "Como devo proceder caso tenha um item comprado roubado e caso eu tenha o cartão gold?"

docs_relevantes = retriever.invoke(consulta)

for i, doc in enumerate(docs_relevantes, start=1):
    print(f"\n--- Documento {i} ---")
    print(doc.metadata)
    print(doc.page_content[:800])

NameError: name 'retriever' is not defined

In [14]:
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0.3
)

In [15]:
prompt = ChatPromptTemplate.from_template("""
Você é um assistente que responde perguntas com base apenas no contexto abaixo.

Contexto:
{contexto}

Pergunta:
{pergunta}

Responda de forma clara e objetiva. Se a resposta não estiver no contexto, diga que não encontrou a informação nos documentos.
""")

In [16]:
def formatar_contexto(docs):
    return "\n\n".join(doc.page_content for doc in docs)

In [17]:
cadeia_rag = (
    {
        "contexto": retriever | formatar_contexto,
        "pergunta": RunnablePassthrough()
    }
    | prompt
    | llm
    | StrOutputParser()
)

NameError: name 'retriever' is not defined

In [18]:
pergunta = "Como devo proceder caso tenha um item comprado roubado e caso eu tenha o cartão gold?"

resposta = cadeia_rag.invoke(pergunta)
print(resposta)

NameError: name 'cadeia_rag' is not defined

In [19]:
pergunta_2 = "Quais benefícios eu tenho com o cartão platinum em viagens?"
resposta_2 = cadeia_rag.invoke(pergunta_2)
print(resposta_2)

NameError: name 'cadeia_rag' is not defined

formato memrag.py

In [20]:
from dotenv import load_dotenv
import os

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

load_dotenv()

def formatar_contexto(docs):
    return "\n\n".join(doc.page_content for doc in docs)

def main():
    arquivos_pdf = [
        "documentos/GTB_standard_Nov23.pdf",
        "documentos/GTB_gold_Nov23.pdf",
        "documentos/GTB_platinum_Nov23.pdf",
    ]

    documentos = []
    for caminho in arquivos_pdf:
        loader = PyPDFLoader(caminho)
        documentos.extend(loader.load())

    splitter = RecursiveCharacterTextSplitter(
        chunk_size=1000,
        chunk_overlap=200
    )
    chunks = splitter.split_documents(documentos)

    embeddings = GoogleGenerativeAIEmbeddings(
        model="models/gemini-embedding-001"
    )

    vectorstore = FAISS.from_documents(chunks, embeddings)
    retriever = vectorstore.as_retriever(search_kwargs={"k": 4})

    llm = ChatGoogleGenerativeAI(
        model="gemini-2.5-flash",
        temperature=0.3
    )

    prompt = ChatPromptTemplate.from_template("""
    Você é um assistente que responde perguntas com base apenas no contexto abaixo.

    Contexto:
    {contexto}

    Pergunta:
    {pergunta}

    Responda de forma clara e objetiva. Se a resposta não estiver no contexto, diga que não encontrou a informação nos documentos.
    """)

    cadeia_rag = (
        {
            "contexto": retriever | formatar_contexto,
            "pergunta": RunnablePassthrough()
        }
        | prompt
        | llm
        | StrOutputParser()
    )

    pergunta = "Como devo proceder caso tenha um item comprado roubado e caso eu tenha o cartão gold?"
    resposta = cadeia_rag.invoke(pergunta)

    print("\nPergunta:")
    print(pergunta)
    print("\nResposta:")
    print(resposta)

if __name__ == "__main__":
    main()

GoogleGenerativeAIError: Error embedding content (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/embed_content_free_tier_requests, limit: 100, model: gemini-embedding-1.0\nPlease retry in 59.00129346s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/embed_content_free_tier_requests', 'quotaId': 'EmbedContentRequestsPerMinutePerUserPerProjectPerModel-FreeTier', '

In [23]:
!pip install -q langchain langchain-community langchain-google-genai pypdf faiss-cpu sentence-transformers

In [24]:
from langchain_community.embeddings import HuggingFaceEmbeddings

In [25]:
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

/tmp/ipykernel_2380/1474760240.py:1: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [26]:
vectorstore = FAISS.from_documents(
    documents=chunks,
    embedding=embeddings
)

In [27]:
retriever = vectorstore.as_retriever(
    search_kwargs={"k": 4}
)

In [28]:
consulta = "Como devo proceder caso tenha um item comprado roubado e caso eu tenha o cartão gold?"

docs_relevantes = retriever.invoke(consulta)

for i, doc in enumerate(docs_relevantes, start=1):
    print(f"\n--- Documento {i} ---")
    print(doc.metadata)
    print(doc.page_content[:800])


--- Documento 1 ---
{'producer': 'Microsoft® Word for Microsoft 365', 'creator': 'Microsoft® Word for Microsoft 365', 'creationdate': '2023-10-26T10:05:20-03:00', 'msip_label_df2f77bf-ac71-4d31-be38-cc6a5f811e56_enabled': 'true', 'msip_label_df2f77bf-ac71-4d31-be38-cc6a5f811e56_setdate': '2023-06-21T14:24:39Z', 'msip_label_df2f77bf-ac71-4d31-be38-cc6a5f811e56_method': 'Privileged', 'msip_label_df2f77bf-ac71-4d31-be38-cc6a5f811e56_name': 'Restricted', 'msip_label_df2f77bf-ac71-4d31-be38-cc6a5f811e56_siteid': 'f06fa858-824b-4a85-aacb-f372cfdc282e', 'msip_label_df2f77bf-ac71-4d31-be38-cc6a5f811e56_actionid': '8050f60f-7cb7-4b1b-99e0-7f18410feb72', 'msip_label_df2f77bf-ac71-4d31-be38-cc6a5f811e56_contentbits': '0', 'title': 'GTB WE', 'author': 'Zachary A. Cardoza', 'moddate': '2023-10-26T10:05:20-03:00', 'source': '/content/documentos/GTB_gold_Nov23.pdf', 'total_pages': 24, 'page': 17, 'page_label': '18'}
* É possível que seja solicitado ao Portador de Cartão que envie o item ou itens dan

In [29]:
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0.3
)

In [30]:
prompt = ChatPromptTemplate.from_template("""
Você é um assistente que responde perguntas com base apenas no contexto abaixo.

Contexto:
{contexto}

Pergunta:
{pergunta}

Responda de forma clara e objetiva. Se a resposta não estiver no contexto, diga que não encontrou a informação nos documentos.
""")

In [31]:
def formatar_contexto(docs):
    return "\n\n".join(doc.page_content for doc in docs)

In [32]:
cadeia_rag = (
    {
        "contexto": retriever | formatar_contexto,
        "pergunta": RunnablePassthrough()
    }
    | prompt
    | llm
    | StrOutputParser()
)

In [33]:
pergunta = "Como devo proceder caso tenha um item comprado roubado e caso eu tenha o cartão gold?"

resposta = cadeia_rag.invoke(pergunta)
print(resposta)

Caso seu item comprado com o cartão Mastercard Gold seja roubado, você deve proceder da seguinte forma:

1.  **Fornecer o recibo original:** Apresentar um recibo original que comprove que o pagamento do item foi feito integralmente com o cartão Mastercard Gold.
2.  **Apresentar laudo de ocorrência policial:** Fornecer, dentro do prazo exigido, cópias oficiais do laudo de ocorrência policial referente ao incidente de roubo.

Além disso, observe que:
*   Ocasionalmente, poderão ser solicitadas informações adicionais para o processamento do seu sinistro, e é sua responsabilidade informar esses dados.
*   É possível que seja solicitado o envio do item danificado (se aplicável, embora no caso de roubo o item geralmente não esteja disponível para envio), às suas custas, para melhor avaliação da reivindicação.


In [34]:
pergunta_2 = "Quais benefícios eu tenho com o cartão platinum em viagens?"
resposta_2 = cadeia_rag.invoke(pergunta_2)
print(resposta_2)

Com o cartão Mastercard Platinum™, você tem acesso a um "Bilhete de Seguro Viagem".

Para obter cobertura em viagens:
*   É imprescindível emitir o Bilhete de Seguro através do portal www.aig.com/Mastercard/pt.
*   O Bilhete de Seguro Viagem tem vigência de 12 (doze) meses a partir da data da emissão.
*   Somente estarão cobertas "Viagens Cobertas" ocorridas após a emissão do Bilhete de Seguro.
*   As coberturas se aplicam se a "Perda Coberta" ocorrer durante a vigência do Bilhete de Seguro.
*   O custo total da passagem de um "Transporte Público Autorizado" deve ser cobrado do seu cartão Mastercard Platinum™ elegível e/ou adquirida com pontos ganhos em um Programa de Recompensas associado ao seu cartão Mastercard Platinum™.
*   Para ser elegível à cobertura, o portador de cartão deve pagar todos os impostos, custos de envio e manuseio relacionados e quaisquer outras taxas exigidas pelo seu cartão Mastercard Platinum™ e/ou pontos ganhos através de um Programa de Recompensas associado a